# 03 Risk Drawdowns
Question: How does strategy risk manifest across macro regimes?

**Warning:** This notebook is descriptive only and must not be used for forecasting, signal generation, or trading decisions.

Drawdowns, volatility, and tail observations shown here are historical diagnostics. They describe how realized return paths behaved under realized macro labels; they are not forecasts, not signals, and not evidence of persistent edge.


In [ ]:
from pathlib import Path
import sys

def _repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing 'src' and 'data'.")

REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.conditioning.regime_join import join_returns_with_regimes
from src.conditioning.regime_stats import risk_summary_by_regime
from src.data.loaders import load_regimes, load_strategy_returns, load_transitions
from src.data.validators import run_all_validations
from src.visualization.plots import plot_drawdown_path

regimes = load_regimes(REPO_ROOT / "data" / "macro" / "regimes_example.csv")
transitions = load_transitions(REPO_ROOT / "data" / "macro" / "transitions_example.csv")
strategy_returns = load_strategy_returns(REPO_ROOT / "data" / "strategies" / "strategy_returns_example.csv")

# Validation is required before any risk or drawdown diagnostics.
run_all_validations(regimes, transitions, strategy_returns)

joined = join_returns_with_regimes(strategy_returns, regimes)
risk_summary = risk_summary_by_regime(joined).sort_values(["strategy_id", "Macro_Regime"]).reset_index(drop=True)
display(risk_summary)

fig = plot_drawdown_path(joined, strategy_id="strat_a")
fig
